import yt_dlp
import whisper
import os
import re
import queue
import torch
from concurrent.futures import ThreadPoolExecutor

# ==========================================================
# 1. CONFIGURATION & USER INPUTS
# ==========================================================
# Add all your YouTube links inside this list
video_links = [
    "Searching the notebook for any explicit YouTube URL values.

Searched for regex `youtube\.com/watch\?v=|youtu\.be/|https://www\.youtube\.com|http://www\.youtube\.com|www\.youtube\.com` (`**/Youtube_Audio_Dataset_Extraction_Pipeline_(YADEP).ipynb`), 100 results

Extracting every YouTube URL string from the notebook contents now.

Ran `pylanceRunCodeSnippet`
Completed with input: {
  "workspaceRoot": "c:\\Users\\abhir\\git-practice",
  "codeSnippet": "import json, re, pathlib\npath = pathlib.Path(r'c:\\Users\\abhir\\git-practice\\Pipeline-to-Gather-Audio-Transcript-Datasets-from-Youtube\\Youtube_Audio_Dataset_Extraction_Pipeline_(YADEP).ipynb')\ntext = path.read_text(encoding='utf-8')\nurls = re.findall(r'https?://(?:www\\.)?youtube\\.com/[A-Za-z0-9_/?=\\-&]+', text)\nurls += re.findall(r'https?://youtu\\.be/[A-Za-z0-9_\\-]+', text)\nurls = sorted(set(urls))\nprint(len(urls))\nfor u in urls:\n    print(u)\n",
  "workingDirectory": "c:\\Users\\abhir\\git-practice",
  "timeout": 30000
}

Here are the YouTube URLs found in the notebook:


Updated todo list",
    # "https://www.youtube.com/watch?v=EXAMPLE_2",
    # "https://www.youtube.com/watch?v=EXAMPLE_3",
]

# Max number of parallel download workers (3 to 5 is ideal for Colab)
MAX_DOWNLOAD_THREADS = 3

# Global thread-safe queue to pass downloaded files to the transcription engine
transcription_queue = queue.Queue()

# Clean video titles to save them cleanly as filenames
def clean_filename(title):
    return re.sub(r'[\\/*?:"<>|]', "", title).replace(" ", "_")

# ==========================================================
# 2. THE DOWNLOAD WORKER FUNCTION (Runs in Parallel)
# ==========================================================
def download_audio_worker(url):
    """
    This function will be assigned to a background thread.
    Multiple instances of this function run simultaneously to download videos.
    """
    ydl_opts = {
        'format': 'bestaudio/best',
        'outtmpl': f'audio_%(id)s.%(ext)s', # Unique temporary filename using YouTube ID
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'mp3',
            'preferredquality': '192',
        }],
        'quiet': True,
    }

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            # Extract video metadata
            info_dict = ydl.extract_info(url, download=False)
            video_title = info_dict.get('title', 'Untitled_Video')
            video_id = info_dict.get('id', 'unknown')
            safe_title = clean_filename(video_title)

            print(f"[DOWNLOAD START] -> Downloading: '{video_title}'...")
            ydl.download([url])

            expected_file = f"audio_{video_id}.mp3"

            if os.path.exists(expected_file):
                print(f"[DOWNLOAD FINISHED] -> Ready for AI: '{video_title}'")
                # Push the file data into the processing queue for Whisper
                transcription_queue.put({
                    'file_path': expected_file,
                    'title': safe_title
                })
            else:
                print(f"✗ [DOWNLOAD ERROR] -> File matching failed for ID: {video_id}")

    except Exception as e:
        print(f"✗ [DOWNLOAD FAILED] -> Error processing {url}: {e}")

# ==========================================================
# 3. THE TRANSCRIPTION ENGINE (Runs Sequentially on GPU)
# ==========================================================
def run_transcription_engine(total_files):
    """
    Consumes files from the queue one-by-one as they finish downloading,
    ensuring the GPU is highly efficient and never overloaded.
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"\n[AI SETUP] Loading Whisper 'medium' model onto {device.upper()}...")
    model = whisper.load_model("medium", device=device)
    print("[AI SETUP] Whisper engine active and waiting for downloads.\n" + "="*60)

    processed_count = 0

    while processed_count < total_files:
        try:
            # Wait for a file to show up in the queue (blocks thread if queue is empty)
            task = transcription_queue.get(timeout=300)
            file_path = task['file_path']
            title = task['title']

            processed_count += 1
            print(f"\n[AI PROCESSING ({processed_count}/{total_files})] -> Transcribing: '{title}'")

            # Run Whisper (forced to Hindi language mode)
            result = model.transcribe(file_path, language="hi")

            # Save the final text output
            output_filename = f"{title}_parallel_transcript.txt"
            with open(output_filename, "w", encoding="utf-8") as f:
                f.write(result["text"])

            print(f"✓ [SAVED] -> Saved transcript to: '{output_filename}'")

            # Delete local temporary audio file to preserve disk space
            if os.path.exists(file_path):
                os.remove(file_path)

            transcription_queue.task_done()

        except queue.Empty:
            print("[AI TIMEOUT] No audio files received within the last 5 minutes. Shutting down pipeline.")
            break
        except Exception as e:
            print(f"✗ [AI ERROR] Failed transcribing {title}: {e}")
            transcription_queue.task_done()

# ==========================================================
# 4. EXECUTION CONTROLLER
# ==========================================================
if __name__ == "__main__":
    total_videos = len(video_links)
    print(f"Starting pipeline for {total_videos} video(s)...")

    # 1. Spin up background threads to handle parallel downloads
    with ThreadPoolExecutor(max_workers=MAX_DOWNLOAD_THREADS) as executor:
        # Map the download function to all URLs across active threads
        download_futures = executor.map(download_audio_worker, video_links)

        # 2. Simultaneously start processing audio files as they arrive in the queue
        run_transcription_engine(total_files=total_videos)

    print("\n" + "="*60 + "\nAll parallel downloads and transcriptions are complete!")

In [ ]:
import sys
!{sys.executable} -m pip install yt-dlp==2024.7.25
!{sys.executable} -m pip install git+https://github.com/openai/whisper.git

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 116.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.4/194.4 kB 20.5 MB/s eta 0:00:00
  Cloning https://github.com/openai/whisper.git to /tmp/pip-req-build-hwltbg65
  Running command git clone --filter=blob:none --quiet https://github.com/openai/whisper.git /tmp/pip-req-build-hwltbg65
  Resolved https://github.com/openai/whisper.git to commit 04f449b8a437f1bbd3dba5c9f826aca972e7709a
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=a944ea6fac9d7bf25b277aa30f7003f9e8a55fb873031f8ac9f52bfe33a97f9e
  Stored in directory: /tmp/pip-ephem-wheel-cache-cu111ok7/wheels/c3/03/25/5e0ba78bc27a3a089f137c9f1d92fdfce16d06996c071a016c
Successfully built openai-whi

In [ ]:
!apt update && apt install -y ffmpeg

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,703 kB]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,038 kB]
Hit:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,297 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-securi

In [ ]:
# Install yt-dlp to download the video/audio and Whisper for ASR
!pip install -q yt-dlp openai-whisper

In [ ]:
import yt_dlp
import whisper
import os

# 1. Define the YouTube URL
video_links = [
    "https://www.youtube.com/watch?v=JLwhLr4ZVd0" # New horticulture video link
]
# 2. Configure audio extraction options
ydl_opts = {
    'format': 'bestaudio/best',
    'outtmpl': 'audio_source.%(ext)s',
    'postprocessors': [{
        'key': 'FFmpegExtractAudio',
        'preferredcodec': 'mp3',
        'preferredquality': '192',
    }],
}

print("Downloading and extracting audio from the video...")
with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    ydl.download([video_url])

# 3. Verify file path (yt-dlp forces the extension specified in postprocessors)
audio_file = "audio_source.mp3"

if os.path.exists(audio_file):
    print("Audio successfully extracted. Loading Whisper Model...")

    # Using the 'small' or 'medium' model balances speed and excellent transcription for Hindi/Indian accents
    # If accuracy needs a further boost, you can swap "small" for "medium" or "large"
    model = whisper.load_model("small")

    print("Transcribing... This may take a few minutes depending on video length.")
    # setting language="hi" forces Hindi script output; remove it or set task="translate" if you want English text.
    result = model.transcribe(audio_file, language="hi", verbose=False)

    # 4. Save the full transcript to a text file
    transcript_path = "krishi_darshan_transcript.txt"
    with open(transcript_path, "w", encoding="utf-8") as f:
        f.write(result["text"])

    print(f"\nTranscription complete! Saved to '{transcript_path}'.")
    print("\n--- Snippet of the Transcript ---")
    print(result["text"][:1000] + "...") # Displays the first 1000 characters
else:
    print("Error: Audio file extraction failed.")

NameError: name 'video_url' is not defined

In [ ]:
import yt_dlp
import whisper
import os

# 1. Define the YouTube URL (Changed from a list to a single string for this script)
video_url = "https://www.youtube.com/watch?v=-L1Y5uZK2xk" # <-- **IMPORTANT**: Replace with a new, working YouTube video URL

# 2. Configure audio extraction options
ydl_opts = {
    'format': 'bestaudio/best',
    'outtmpl': 'audio_source.%(ext)s',
    'postprocessors': [{
        'key': 'FFmpegExtractAudio',
        'preferredcodec': 'mp3',
        'preferredquality': '192',
    }],
    'quiet': False # Shows the download progress bar
}

print("Downloading and extracting audio from the video...")
with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    # Now it properly passes the 'video_url' variable defined on Line 6
    ydl.download([video_url])

# 3. Verify file path
audio_file = "audio_source.mp3"

if os.path.exists(audio_file):
    print("\nAudio successfully extracted. Loading Whisper Model...")

    # Swapped from 'small' to 'medium' for vastly better accuracy on this specific Haryana horticulture broadcast
    model = whisper.load_model("medium")

    print("Transcribing... This may take a few minutes depending on video length.")
    result = model.transcribe(audio_file, language="hi", verbose=False)

    # 4. Save the full transcript to a text file
    transcript_path = "krishi_darshan_transcript.txt"
    with open(transcript_path, "w", encoding="utf-8") as f:
        f.write(result["text"])

    print(f"\n✓ Transcription complete! Saved to '{transcript_path}'.")
    print("\n--- Snippet of the Transcript ---")
    print(result["text"][:1000] + "...")
else:
    print("Error: Audio file extraction failed.")

ModuleNotFoundError: No module named 'yt_dlp'

In [ ]:
# Install yt-dlp to download the video/audio and Whisper for ASR
!pip install -q yt-dlp openai-whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 7.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 32.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 106.4 MB/s eta 0:00:00


In [ ]:
import yt_dlp
import whisper
import os

# 1. Define the YouTube URL
video_url = "https://www.youtube.com/watch?v=-L1Y5uZK2xk"

# 2. Configure audio extraction options
ydl_opts = {
    'format': 'bestaudio/best',
    'outtmpl': 'audio_source.%(ext)s',
    'postprocessors': [{
        'key': 'FFmpegExtractAudio',
        'preferredcodec': 'mp3',
        'preferredquality': '192',
    }],
}

print("Downloading and extracting audio from the video...")
with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    ydl.download([video_url])

# 3. Verify file path (yt-dlp forces the extension specified in postprocessors)
audio_file = "audio_source.mp3"

if os.path.exists(audio_file):
    print("Audio successfully extracted. Loading Whisper Model...")

    # Using the 'small' or 'medium' model balances speed and excellent transcription for Hindi/Indian accents
    # If accuracy needs a further boost, you can swap "small" for "medium" or "large"
    model = whisper.load_model("small")

    print("Transcribing... This may take a few minutes depending on video length.")
    # setting language="hi" forces Hindi script output; remove it or set task="translate" if you want English text.
    result = model.transcribe(audio_file, language="hi", verbose=False)

    # 4. Save the full transcript to a text file
    transcript_path = "krishi_darshan_transcript.txt"
    with open(transcript_path, "w", encoding="utf-8") as f:
        f.write(result["text"])

    print(f"\nTranscription complete! Saved to '{transcript_path}'.")
    print("\n--- Snippet of the Transcript ---")
    print(result["text"][:1000] + "...") # Displays the first 1000 characters
else:
    print("Error: Audio file extraction failed.")

[youtube] Extracting URL: https://www.youtube.com/watch?v=-L1Y5uZK2xk
[youtube] -L1Y5uZK2xk: Downloading webpage


[youtube] -L1Y5uZK2xk: Downloading android vr player API JSON
[info] -L1Y5uZK2xk: Downloading 1 format(s): 140
[download] Destination: audio_source.m4a
[download] 100% of   20.85MiB in 00:00:02 at 7.35MiB/s   
[FixupM4a] Correcting container of "audio_source.m4a"
[ExtractAudio] Destination: audio_source.mp3
Deleting original file audio_source.m4a (pass -k to keep)
Audio successfully extracted. Loading Whisper Model...


100%|████████████████████████████████████████| 461M/461M [00:02<00:00, 180MiB/s]


Transcribing... This may take a few minutes depending on video length.


 98%|█████████▊| 132054/135054 [10:23<00:14, 211.63frames/s]


Transcription complete! Saved to 'krishi_darshan_transcript.txt'.

--- Snippet of the Transcript ---
 выпускinished mmm अब इनका औडपादन अच्छा बना है इसके लिए समचामाइ कारे के अन्टर गत हम कुझ कुझ कारे कर सकते हैं ताका उडपादन अच्छा हो आर किसानों को लाभ मेले. निवुबर्गी फल्ड ज़ेसा कि आप जानते हो ये एक फलों का समू है, इस में कई प्रकार के फला आते है, तो इनकी सब से पहली विसिस्टा तो ये किसान बाई इनको लगा कर बर सुवर आम्दनी प्राप्त कर सकते हैं, अगर इसका पोसक्ततों के फिल्षाप से इसका व ये निवुबर्गी पले ये ये ये प्रत्याख्सी कारा कोतें, तो वो सब सरीर की रोग रोग रोदक शम्टा में कापी बडोतरी होती हैं, और भिसेस तोर से जो वागर गरे प्रुट वो चोकोतरा है, ये ताएप तु डाविटिक आप निंटिट करने में कापी कामयाब अदिक सदिक आप को उपष मिले, जिस से उपोकता हो तक आप पादिक से अदिक ये पल पहुषा असक तने वी पलो के हम ने नाम लिएं, इन में पानि की मात्रा बहुज जाडा होती, अई गनी की रस बहुज जाडा होता है, तो इन में अगर इस समया, जब आरिस कदम हो चुकिये, अगर हमारी जमीनो में पानि नहीं रहेगा, तो पलो से पानि बापस आने लगता है, जिस से पलो की गु